# Gemini LLM response check (plain `requests`)

Verifies Gemini via `credit_risk.gemini_requests.GeminiChat`:
key validation (`GET /models`), plain `invoke`, and the `bind_tools`
path that `tool_loop.py` uses. Run top to bottom.

In [ ]:
import os
import dotenv

dotenv.load_dotenv()

key = os.getenv("GEMINI_API_KEY", "")
print("GEMINI_API_KEY:", "set (" + key[:4] + "..." + key[-4:] + ")" if key and key != "your-gemini-api-key-here" else "NOT SET")
print("LLM_MODEL_NAME:", os.getenv("LLM_MODEL_NAME", "gemini-2.5-flash"))
print("LLM_ENDPOINT:", repr(os.getenv("LLM_ENDPOINT", "")))

In [ ]:
# Raw requests check — GET /models (validates key + lists usable models)
from credit_risk.gemini_requests import list_gemini_models

models = list_gemini_models(key)
print(f"{len(models)} models")
for m in models:
    print(" -", m)

In [ ]:
from credit_risk.config import get_llm

try:
    llm = get_llm()
except RuntimeError as e:
    print("LLM init failed:")
    print(e)
    raise

print("llm type:", type(llm).__name__)
print("llm model:", getattr(llm, "model", getattr(llm, "model_name", "unknown")))

resp = llm.invoke("Reply with exactly: GEMINI_OK")
print("response type:", type(resp).__name__)
print("content:", resp.content)

In [ ]:
# bind_tools path — the same interface tool_loop.py relies on
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def search_pages(query: str) -> str:
    """Fake page search for smoke test."""
    return f"hit for {query}"


bound = llm.bind_tools([search_pages])
out = bound.invoke([HumanMessage(content="Use search_pages with query 'revenue' once, then stop.")])
print("tool_calls:", out.tool_calls)
print("content:", repr(out.content))

In [ ]:
# Override path (fresh instance, bypasses singleton cache)
from credit_risk.config import get_llm

llm_small = get_llm(max_tokens=64)
resp = llm_small.invoke("What is 2+2? Answer with the number only.")
print("content:", resp.content)